# 23 Build raw same-month panel

目标：首次汇合基金、持仓与股票三分支，构建同一形成月的 59 特征面板。

- 输入：21、22、06 号输出
- 正式输出：raw_same_month_panel.parquet、三分支连接审计
- 主键：wficn+month
- 上游依赖：06、21、22
- smoke 输出与正式输出隔离；报告会明确标注运行模式。

In [ ]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config_utils import load_project_config
from src.path_utils import find_project_root
from src.runtime_paths import (
    clean_output_path,
    execution_context,
    install_clean_read_guard,
)

ROOT = find_project_root(ROOT)
EXECUTION_CONTEXT = execution_context()
install_clean_read_guard(ROOT)
CONFIG = load_project_config(ROOT)
PARAMS = CONFIG["parameters"]
PATHS = CONFIG["resolved_paths"]
RUN_MODE = os.getenv("KANIEL_RUN_MODE", "smoke").strip().lower()
SMOKE = RUN_MODE != "full"
if RUN_MODE == "full":
    FULL_RUN_TMP = (
        clean_output_path(ROOT, "data/interim/_full_run_tmp")
        if EXECUTION_CONTEXT == "clean_rebuild_compute"
        else PATHS["full_run_tmp"]
    )
    PATHS["full_run_tmp"] = FULL_RUN_TMP
    FULL_RUN_TMP.mkdir(parents=True, exist_ok=True)
    os.environ["TMP"] = str(FULL_RUN_TMP)
    os.environ["TEMP"] = str(FULL_RUN_TMP)
    os.environ["TMPDIR"] = str(FULL_RUN_TMP)
    os.environ["POLARS_TEMP_DIR"] = str(FULL_RUN_TMP)
STARTED_AT = time.time()
print({
    "project_root": str(ROOT),
    "run_mode": RUN_MODE,
    "execution_context": EXECUTION_CONTEXT,
})


In [ ]:
import pandas as pd
from src.atomic_write import write_csv_atomic, write_parquet_atomic
from src.macro_data import build_monthly_macro_features
from src.runtime_paths import output_path, upstream_path

stock=pd.read_parquet(upstream_path(ROOT,RUN_MODE,"21_aggregate_stock_features_to_funds","data/processed/fund/fund_46_stock_features.parquet"))
fund=pd.read_parquet(upstream_path(ROOT,RUN_MODE,"22_rank_fund_and_family_features","data/processed/fund/fund_13_ranked_features.parquet"))
alpha_source=pd.read_parquet(upstream_path(ROOT,RUN_MODE,"06_compute_fund_rret_and_momentum","data/processed/fund/fund_alpha_momentum_raw.parquet"))
for frame in [stock,fund,alpha_source]:
    frame["month"]=pd.to_datetime(frame["month"]).dt.to_period("M").dt.to_timestamp()

macro,audits=build_monthly_macro_features(
    PATHS["sentiment"],
    PATHS["cfnai"],
    sentiment_sheet=CONFIG["path_options"]["sentiment"]["sheet_name"],
)
assert list(macro.columns)==["observation_month","Sentiment","CFNAI"]
assert not macro.duplicated(["observation_month"]).any()
assert macro["observation_month"].is_monotonic_increasing
assert "SENT_ORTH" not in macro.columns
assert not (macro[["Sentiment","CFNAI"]]==float(PARAMS["missing_export_value"])).any().any()

before=len(fund)
panel=fund.merge(stock,on=["wficn","month"],how="inner",validate="one_to_one")
after_stock=len(panel)
panel=panel.merge(
    alpha_source[["wficn","month","alpha"]],
    on=["wficn","month"],
    how="left",
    validate="one_to_one",
)
raw_macro=macro.rename(columns={
    "observation_month":"month",
    "Sentiment":"Sentiment_observation_month_value",
    "CFNAI":"CFNAI_observation_month_value",
})
panel=panel.merge(raw_macro,on="month",how="left",validate="many_to_one")
names=['r2_1', 'r12_2', 'r12_7', 'r36_13', 'ST_Rev', 'LT_Rev', 'Investment', 'NOA', 'DPI2A', 'NI', 'PROF', 'ATO', 'CTO', 'FC2Y', 'OP', 'PM', 'RNA', 'ROA', 'ROE', 'SGA2S', 'D2A', 'AC', 'OA', 'OL', 'PCM', 'A2ME', 'BEME', 'C', 'CF', 'CF2P', 'D2P', 'E2P', 'Q', 'S2P', 'Lev', 'AT', 'Beta', 'IdioVol', 'LME', 'LTurnover', 'MktBeta', 'Rel2High', 'Resid_Var', 'Spread', 'SUV', 'Variance', 'F_ST_Rev', 'F_r2_1', 'F_r12_2', 'age', 'tna', 'flow', 'exp_ratio', 'turnover', 'Family_TNA', 'fund_no', 'Family_r12_2', 'Family_age', 'Family_flow']
for name in names:
    if name not in panel:
        panel[name]=float("nan")
panel=panel[[
    "wficn","month",*names,"alpha",
    "Sentiment_observation_month_value","CFNAI_observation_month_value",
]]
assert len(names)==59 and len(set(names))==59
assert not panel.duplicated(["wficn","month"]).any()
assert not {"Sentiment","CFNAI","alpha"} & set(names)
assert "rret" not in panel.columns
assert "official_rret_raw" not in panel.columns

join_audit=pd.DataFrame([
    {"join":"fund_to_fund_stock","left_rows":before,"result_rows":after_stock,
     "unmatched_left":before-after_stock,
     "expansion_ratio":after_stock/before if before else 1.0},
    {"join":"add_same_month_project_alpha","left_rows":after_stock,"result_rows":len(panel),
     "unmatched_left":int(panel["alpha"].isna().sum()),
     "expansion_ratio":len(panel)/after_stock if after_stock else 1.0},
    {"join":"raw_observation_month_macro_audit_only","left_rows":after_stock,
     "result_rows":len(panel),
     "unmatched_left":int(panel[["Sentiment_observation_month_value","CFNAI_observation_month_value"]].isna().all(axis=1).sum()),
     "expansion_ratio":len(panel)/after_stock if after_stock else 1.0},
])

macro_out=output_path(ROOT,RUN_MODE,"23_build_raw_same_month_panel","data/processed/macro/monthly_macro_features.parquet")
panel_out=output_path(ROOT,RUN_MODE,"23_build_raw_same_month_panel","outputs/panels/raw_same_month_panel.parquet")
join_out=output_path(ROOT,RUN_MODE,"23_build_raw_same_month_panel","data/audit/23_three_branch_join.csv")
coverage_out=output_path(ROOT,RUN_MODE,"23_build_raw_same_month_panel","data/audit/macro_month_coverage.csv")
duplicate_out=output_path(ROOT,RUN_MODE,"23_build_raw_same_month_panel","data/audit/macro_duplicate_months.csv")
missing_out=output_path(ROOT,RUN_MODE,"23_build_raw_same_month_panel","data/audit/macro_missing_months.csv")
hash_out=output_path(ROOT,RUN_MODE,"23_build_raw_same_month_panel","data/audit/macro_input_hashes.csv")
write_parquet_atomic(macro,macro_out)
write_parquet_atomic(panel,panel_out)
write_csv_atomic(join_audit,join_out)
write_csv_atomic(audits["coverage"],coverage_out)
write_csv_atomic(audits["duplicate_months"],duplicate_out)
write_csv_atomic(audits["missing_months"],missing_out)
write_csv_atomic(audits["input_hashes"],hash_out)

INPUTS_USED=[
    "fund_46_stock_features.parquet","fund_13_ranked_features.parquet",
    "fund_alpha_momentum_raw.parquet",
    str(PATHS["sentiment"].relative_to(ROOT)),str(PATHS["cfnai"].relative_to(ROOT)),
]
OUTPUTS_WRITTEN=[str(path.relative_to(ROOT)) for path in [
    macro_out,panel_out,join_out,coverage_out,duplicate_out,missing_out,hash_out,
]]
INPUT_COUNTS={"fund_branch":before,"stock_holdings_branch":len(stock)}
OUTPUT_COUNTS={"raw_panel":len(panel),"macro_months":len(macro)}
PRIMARY_KEY=["wficn","month"]; DUPLICATE_KEY_COUNT=0
EXCLUDED_ROW_COUNT=before-after_stock; EXCEPTION_FILE=str(join_out.relative_to(ROOT))
STAGE_STATUS="passed"
STAGE_NOTES=[
    "Raw macro columns are observation-month audit values, not model-ready aligned values.",
    "Sentiment and CFNAI production lags are applied separately in Notebook 24.",
]


In [ ]:
from src.audit_utils import write_run_summary
summary_path = write_run_summary(
    ROOT, "23_build_raw_same_month_panel", stage="23", started_at=STARTED_AT,
    input_files=INPUTS_USED, output_files=OUTPUTS_WRITTEN,
    input_row_counts=INPUT_COUNTS, output_row_counts=OUTPUT_COUNTS,
    primary_key=PRIMARY_KEY, duplicate_key_count=DUPLICATE_KEY_COUNT,
    excluded_row_count=EXCLUDED_ROW_COUNT, exception_file=EXCEPTION_FILE,
    status=STAGE_STATUS, notes=STAGE_NOTES,
)
print(summary_path)
